In [1]:
import pandas as pd

In [2]:
dataset=pd.read_csv("feature_selected_data.csv")

In [3]:
dataset

,vehicle_mass_kg,test_mass_kg,fuel_type,engine_capacity_cc,engine_power_kw,fuel_efficiency_km_per_l
0,1670.0,1782.0,petrol,2487.0,131.0,18.181818
1,1493.0,1576.0,petrol,1199.0,96.0,16.666667
2,1649.0,1814.0,petrol,1598.0,132.0,17.241379
3,1560.0,1640.0,petrol,1987.0,112.0,19.230769
4,1290.0,1427.0,petrol,999.0,81.0,16.129032
...,...,...,...,...,...,...
6528924,1808.0,2140.0,diesel,1997.0,106.0,13.513514
6528925,1475.0,1574.0,petrol,1798.0,72.0,21.276596
6528926,1475.0,1569.0,petrol,1798.0,72.0,21.276596
6528927,1808.0,2140.0,diesel,1997.0,106.0,13.513514


In [4]:
df_sample = dataset.sample(n=25_000, random_state=42)


In [5]:
df_sample

,vehicle_mass_kg,test_mass_kg,fuel_type,engine_capacity_cc,engine_power_kw,fuel_efficiency_km_per_l
2212922,1393.0,1514.0,petrol,1498.0,110.0,16.393443
140455,1575.0,1695.0,petrol,1469.0,96.0,16.666667
1973553,1649.0,1756.0,petrol,1598.0,132.0,17.857143
4289829,1242.0,1270.0,petrol,999.0,81.0,17.857143
1146547,1502.0,1616.0,petrol,1998.0,137.0,16.393443
...,...,...,...,...,...,...
4836961,1065.0,1165.0,petrol,999.0,52.0,20.833333
3274016,1481.0,1598.0,petrol,1998.0,110.0,15.625000
2452726,1161.0,1262.0,petrol,999.0,67.0,19.230769
2152468,2075.0,2159.0,petrol,1998.0,167.0,12.738854


In [7]:
final_features = [
    'engine_power_kw',
    'vehicle_mass_kg',
    'test_mass_kg',
    'engine_capacity_cc',
    'fuel_type'
]

target_column = 'fuel_efficiency_km_per_l'  # replace with your actual target

X = df_sample[final_features]
y = df_sample[target_column]


In [11]:
X_encoded = pd.get_dummies(X, drop_first=True)


In [12]:
X_encoded = X_encoded.astype(int)


In [13]:
X_encoded 

,engine_power_kw,vehicle_mass_kg,test_mass_kg,engine_capacity_cc,fuel_type_petrol
2212922,110,1393,1514,1498,1
140455,96,1575,1695,1469,1
1973553,132,1649,1756,1598,1
4289829,81,1242,1270,999,1
1146547,137,1502,1616,1998,1
...,...,...,...,...,...
4836961,52,1065,1165,999,1
3274016,110,1481,1598,1998,1
2452726,67,1161,1262,999,1
2152468,167,2075,2159,1998,1


In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)


In [15]:
X_train

,engine_power_kw,vehicle_mass_kg,test_mass_kg,engine_capacity_cc,fuel_type_petrol
1876587,131,1545,1648,1998,1
5101523,167,2133,2287,3124,1
75028,96,1592,1748,1199,1
5095337,145,1688,1820,1969,1
5235951,84,1262,1367,999,1
...,...,...,...,...,...
68713,96,1436,1526,1499,0
2975302,88,1365,1463,999,1
5503826,167,1975,2191,2967,0
4794298,96,1567,1667,1499,0


In [16]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [20]:
X_train_scaled

array([[ 1.03788328,  0.41099205,  0.34244535,  0.97014425,  0.56372288],
       [ 2.17524115,  2.56148464,  2.57256164,  3.29104107,  0.56372288],
       [-0.06788131,  0.58288517,  0.69144633, -0.67674435,  0.56372288],
       ...,
       [ 2.17524115,  1.98363119,  2.2375207 ,  2.96743467, -1.77392125],
       [-0.06788131,  0.49145266,  0.40875554, -0.05838818, -1.77392125],
       [-0.66815352,  0.06720582,  0.10861469,  0.10856799,  0.56372288]],
      shape=(20000, 5))

In [17]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

rf = RandomForestRegressor(random_state=42)

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

grid_search = GridSearchCV(
    rf,
    param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)


,estimator,RandomForestR...ndom_state=42)
,param_grid,"{'max_depth': [None, 10, ...], 'min_samples_leaf': [1, 2], 'min_samples_split': [2, 5], 'n_estimators': [100, 200]}"
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,3
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,200


In [22]:
best_model = grid_search.best_estimator_

print("Best parameters:", grid_search.best_params_)

from sklearn.metrics import r2_score

y_pred = best_model.predict(X_test_scaled)
r2 = r2_score(y_test, y_pred)

print("Final R² score:", r2)


Best parameters: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}
Final R² score: 0.9176238297505097


In [23]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV

gbr = GradientBoostingRegressor(random_state=42)

gbr_param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5],
    'subsample': [0.8, 1.0]
}


In [24]:
gbr_grid = GridSearchCV(
    gbr,
    gbr_param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1
)

gbr_grid.fit(X_train_scaled, y_train)


,estimator,GradientBoost...ndom_state=42)
,param_grid,"{'learning_rate': [0.05, 0.1], 'max_depth': [3, 5], 'n_estimators': [100, 200], 'subsample': [0.8, 1.0]}"
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,3
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,loss,'squared_error'


In [25]:
from sklearn.metrics import r2_score

gbr_best = gbr_grid.best_estimator_

y_pred_gbr = gbr_best.predict(X_test_scaled)
print("Gradient Boosting R²:", r2_score(y_test, y_pred_gbr))
print("Best GBR params:", gbr_grid.best_params_)


Gradient Boosting R²: 0.8994358572899758
Best GBR params: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.8}


In [26]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    objective='reg:squarederror',
    random_state=42
)

xgb_param_grid = {
    'n_estimators': [200, 300],
    'learning_rate': [0.05, 0.1],
    'max_depth': [4, 6],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}


In [27]:
xgb_grid = GridSearchCV(
    xgb,
    xgb_param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1
)

xgb_grid.fit(X_train_scaled, y_train)


,estimator,"XGBRegressor(...ree=None, ...)"
,param_grid,"{'colsample_bytree': [0.8, 1.0], 'learning_rate': [0.05, 0.1], 'max_depth': [4, 6], 'n_estimators': [200, 300], ...}"
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,3
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,objective,'reg:squarederror'


In [28]:
xgb_best = xgb_grid.best_estimator_

y_pred_xgb = xgb_best.predict(X_test_scaled)
print("XGBoost R²:", r2_score(y_test, y_pred_xgb))
print("Best XGB params:", xgb_grid.best_params_)


XGBoost R²: 0.911366626726715
Best XGB params: {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 300, 'subsample': 0.8}


In [29]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

y_pred = lr.predict(X_test_scaled)
print("Multiple Linear Regression R²:", r2_score(y_test, y_pred))


Multiple Linear Regression R²: 0.5469098251124489


In [12]:
from sklearn.svm import LinearSVR

linear_svr = LinearSVR(
    C=1.0,
    epsilon=0.1,
    max_iter=5000,
    random_state=42
)

linear_svr.fit(X_train_scaled, y_train)


,epsilon,0.1
,tol,0.0001
,C,1.0
,loss,'epsilon_insensitive'
,fit_intercept,True
,intercept_scaling,1.0
,dual,'auto'
,verbose,0
,random_state,42
,max_iter,5000


In [15]:
from sklearn.metrics import r2_score

y_pred = linear_svr.predict(X_test_scaled)

# 5️⃣ Compute R²
r2 = r2_score(y_test, y_pred)
print("SVR R² score:", r2)


SVR R² score: 0.5370497733644168


In [18]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score

# Define Ridge and hyperparameter grid
ridge = Ridge()
param_grid = {'alpha': [0.01, 0.1, 1, 10, 100]}

# Grid Search with 5-fold cross-validation
grid = GridSearchCV(ridge, param_grid, cv=5, scoring='r2')
grid.fit(X_train_scaled, y_train)

# Best model from GridSearch
best_ridge = grid.best_estimator_

# Predict on test set
y_pred = best_ridge.predict(X_test_scaled)

# R² score
print("Best Ridge alpha:", grid.best_params_['alpha'])
print("Ridge Regression R²:", r2_score(y_test, y_pred))



Best Ridge alpha: 10
Ridge Regression R²: 0.5468754068628611


In [19]:
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score

# Define ElasticNet and hyperparameter grid
elastic = ElasticNet()
param_grid = {
    'alpha': [0.01, 0.1, 1, 10],
    'l1_ratio': [0.1, 0.5, 0.7, 0.9, 1.0]  # l1_ratio=1 is Lasso, 0 is Ridge
}

# Grid Search with 5-fold cross-validation
grid = GridSearchCV(elastic, param_grid, cv=5, scoring='r2')
grid.fit(X_train_scaled, y_train)

# Best model from GridSearch
best_elastic = grid.best_estimator_

# Predict on test set
y_pred = best_elastic.predict(X_test_scaled)

# R² score
print("Best ElasticNet params:", grid.best_params_)
print("ElasticNet R²:", r2_score(y_test, y_pred))


Best ElasticNet params: {'alpha': 0.01, 'l1_ratio': 1.0}
ElasticNet R²: 0.546316073893765


In [20]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score

# Define Decision Tree and hyperparameter grid
dt = DecisionTreeRegressor(random_state=42)
param_grid = {
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Grid Search with 5-fold cross-validation
grid = GridSearchCV(dt, param_grid, cv=5, scoring='r2')
grid.fit(X_train_scaled, y_train)

# Best model from GridSearch
best_dt = grid.best_estimator_

# Predict on test set
y_pred = best_dt.predict(X_test_scaled)

# R² score
print("Best Decision Tree params:", grid.best_params_)
print("Decision Tree R²:", r2_score(y_test, y_pred))


Best Decision Tree params: {'max_depth': None, 'min_samples_leaf': 4, 'min_samples_split': 10}
Decision Tree R²: 0.8967268329790807


In [21]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score

# Define KNN Regressor and hyperparameter grid
knn = KNeighborsRegressor()
param_grid = {
    'n_neighbors': [3, 5, 7, 9],
    'weights': ['uniform', 'distance'],
    'p': [1, 2]  # 1 = Manhattan, 2 = Euclidean
}

# Grid Search with 5-fold cross-validation
grid = GridSearchCV(knn, param_grid, cv=5, scoring='r2')
grid.fit(X_train_scaled, y_train)

# Best model from GridSearch
best_knn = grid.best_estimator_

# Predict on test set
y_pred = best_knn.predict(X_test_scaled)

# R² score
print("Best KNN params:", grid.best_params_)
print("KNN R²:", r2_score(y_test, y_pred))


Best KNN params: {'n_neighbors': 9, 'p': 1, 'weights': 'distance'}
KNN R²: 0.9034562615474517


In [22]:
from sklearn.ensemble import AdaBoostRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score

# Define AdaBoost Regressor and hyperparameter grid
ada = AdaBoostRegressor(random_state=42)
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.5, 1.0],
    'loss': ['linear', 'square', 'exponential']
}

# Grid Search with 5-fold cross-validation
grid = GridSearchCV(ada, param_grid, cv=5, scoring='r2')
grid.fit(X_train_scaled, y_train)

# Best model from GridSearch
best_ada = grid.best_estimator_

# Predict on test set
y_pred = best_ada.predict(X_test_scaled)

# R² score
print("Best AdaBoost params:", grid.best_params_)
print("AdaBoost R²:", r2_score(y_test, y_pred))


Best AdaBoost params: {'learning_rate': 0.1, 'loss': 'exponential', 'n_estimators': 50}
AdaBoost R²: 0.5873472260805273
